# Смерджить новый df со старыми данными.
Для чего: новые данные будут иметь `geonameid` в качестве первичного ключа для городов.

Старые данные: города имели рандомный `id`, который используется как первичный ключ, для данных по климату и тд.

In [2]:
import pandas as pd

In [3]:
df_2023 = pd.read_pickle("./data/2023/numbeo_links.pkl")
df_2026 = pd.read_pickle("./data/numbeo_links_2026.pkl")

In [4]:
df_2026["link"][2]

'https://www.numbeo.com/cost-of-living/in/Basel?displayCurrency=USD'

In [6]:
matched_city_country = set(df_2026["city_country"]) & set(df_2023["city_country"])
count_matched = len(matched_city_country)

In [7]:
count_matched

437

In [10]:
count_rows_matched = df_2026["city_country"].isin(df_2023["city_country"]).sum()
display(count_rows_matched)

np.int64(437)

In [11]:
match_mask = df_2026["city_country"].isin(df_2023["city_country"])

df_2026_matched = df_2026[match_mask].copy()
df_2026_unmatched = df_2026[~match_mask].copy()

In [13]:
print("Совпали:", len(df_2026_matched))
print("Не совпали:", len(df_2026_unmatched))

Совпали: 437
Не совпали: 90


In [14]:
mask_match = df_2026["city_country"].isin(df_2023["city_country"])

df_2026_matched = df_2026[mask_match].copy()

In [15]:
df_2026_unmatched = df_2026[~mask_match].copy()

In [16]:
mask_df_unmatched = ~df_2023["city_country"].isin(df_2026["city_country"])

df_unmatched = df_2023[mask_df_unmatched].copy()

In [17]:
print("Совпали:", len(df_2026_matched))
print("Не совпали в df_2026:", len(df_2026_unmatched))
print("Остались лишние в df_2023:", len(df_unmatched))

Совпали: 437
Не совпали в df_2026: 90
Остались лишние в df_2023: 97


In [18]:
display(df_2026_unmatched[["city_country"]])
display(df_unmatched[["city_country"]])

,city_country
3,"Lugano, Switzerland"
22,"Charleston, SC, United States"
25,"Anchorage, AK, United States"
35,"Arlington, VA, United States"
36,"Aarhus, Denmark"
...,...
522,"Indore, India"
523,"Guwahati, India"
524,"Lucknow (Lakhnau), India"
525,"Rajkot, India"


,city_country
city_id,
1,"Hamilton, Bermuda"
4,"Lucerne, Switzerland"
6,"Zug, Switzerland"
10,"Nassau, Bahamas"
29,"Tromso, Norway"
...,...
521,"New Cairo, Egypt"
525,"Alanya, Turkey"
526,"Konya, Turkey"


In [ ]:
df_unmatched[["city_country"]].head(50)

In [ ]:
df_unmatched[["city_country"]].tail(50)


In [19]:
print(
    "уникальных city_country в df_2026:",
    df_2026["city_country"].nunique(),
)
print("уникальных city_country в df_2023:", df_2023["city_country"].nunique())


уникальных city_country в df_2026: 527
уникальных city_country в df_2023: 534


строим словарь соответствия между `city_country` и `city_id`

In [20]:
city_id_map = df_2023.reset_index().set_index("city_country")["city_id"]

In [21]:
city_id_map

city_country
Hamilton, Bermuda          1
Basel, Switzerland         2
Zurich, Switzerland        3
Lucerne, Switzerland       4
Lausanne, Switzerland      5
                        ... 
Giza, Egypt              530
Alexandria, Egypt        531
Lahore, Pakistan         532
Islamabad, Pakistan      533
Karachi, Pakistan        534
Name: city_id, Length: 534, dtype: int64

In [ ]:
# 1. Копия, чтобы не портить исходник
df_2026_final = df_2026.copy()

In [ ]:
# 2. Пытаемся проставить правильные city_id по совпадению city_country
df_2026_final["new_city_id"] = df_2026_final["city_country"].map(
    city_id_map
)

In [ ]:
# 3. Для несовпавших назначаем новые id, начиная с 535
mask_unmatched = df_2026_final["new_city_id"].isna()
n_unmatched = mask_unmatched.sum()

In [25]:
new_ids = range(535, 535 + n_unmatched)
df_2026_final.loc[mask_unmatched, "new_city_id"] = list(new_ids)

In [ ]:
# 4. Приводим к int и делаем новым индексом
df_2026_final["new_city_id"] = df_2026_final["new_city_id"].astype(int)
df_2026_final = df_2026_final.set_index("new_city_id")

In [ ]:
# 5. Назовем индекс как нужно
df_2026_final.index.name = "city_id"

In [28]:
df_2026_final

,city,state_code,state_name,country_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link
city_id,,,,,,,,,,,,,,,,,,,
3,Zurich,NaN,NaN,CHE,Switzerland,"Zurich, Switzerland",123.9,71.3,99.6,125.6,176.4,204.6,11.9,36.9,81.5,76.7,70.1,25.3,https://www.numbeo.com/cost-of-living/in/Zuric...
8,Geneva,NaN,NaN,CHE,Switzerland,"Geneva, Switzerland",118.4,64.3,93.4,116.0,161.3,200.5,13.0,32.6,82.6,70.5,69.9,24.2,https://www.numbeo.com/cost-of-living/in/Genev...
2,Basel,NaN,NaN,CHE,Switzerland,"Basel, Switzerland",114.0,47.3,83.2,113.7,186.9,206.3,9.8,36.9,82.8,68.9,69.3,31.4,https://www.numbeo.com/cost-of-living/in/Basel...
535,Lugano,NaN,NaN,CHE,Switzerland,"Lugano, Switzerland",113.8,44.2,81.6,109.9,160.3,NaN,NaN,NaN,NaN,78.4,NaN,NaN,https://www.numbeo.com/cost-of-living/in/Lugan...
5,Lausanne,NaN,NaN,CHE,Switzerland,"Lausanne, Switzerland",113.0,52.8,85.2,111.4,181.2,206.4,11.0,33.3,73.3,71.6,70.3,NaN,https://www.numbeo.com/cost-of-living/in/Lausa...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
620,Indore,NaN,NaN,IND,India,"Indore, India",18.1,4.0,11.6,20.5,68.7,NaN,NaN,NaN,NaN,51.8,NaN,60.2,https://www.numbeo.com/cost-of-living/in/Indor...
621,Guwahati,NaN,NaN,IND,India,"Guwahati, India",17.7,3.3,11.0,21.3,62.9,NaN,NaN,NaN,NaN,NaN,NaN,75.8,https://www.numbeo.com/cost-of-living/in/Guwah...
622,Lucknow,NaN,NaN,IND,India,"Lucknow (Lakhnau), India",17.5,3.4,11.0,19.8,69.8,NaN,NaN,NaN,NaN,54.5,62.9,77.9,https://www.numbeo.com/cost-of-living/in/Luckn...


In [ ]:
df_2026_final.head(30)

In [29]:
df_2026_final.to_pickle("./data/matched_df_2026.pkl")

In [30]:
new_dff = pd.read_pickle("./data/matched_df_2026.pkl")

In [31]:
new_dff

,city,state_code,state_name,country_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link
city_id,,,,,,,,,,,,,,,,,,,
3,Zurich,NaN,NaN,CHE,Switzerland,"Zurich, Switzerland",123.9,71.3,99.6,125.6,176.4,204.6,11.9,36.9,81.5,76.7,70.1,25.3,https://www.numbeo.com/cost-of-living/in/Zuric...
8,Geneva,NaN,NaN,CHE,Switzerland,"Geneva, Switzerland",118.4,64.3,93.4,116.0,161.3,200.5,13.0,32.6,82.6,70.5,69.9,24.2,https://www.numbeo.com/cost-of-living/in/Genev...
2,Basel,NaN,NaN,CHE,Switzerland,"Basel, Switzerland",114.0,47.3,83.2,113.7,186.9,206.3,9.8,36.9,82.8,68.9,69.3,31.4,https://www.numbeo.com/cost-of-living/in/Basel...
535,Lugano,NaN,NaN,CHE,Switzerland,"Lugano, Switzerland",113.8,44.2,81.6,109.9,160.3,NaN,NaN,NaN,NaN,78.4,NaN,NaN,https://www.numbeo.com/cost-of-living/in/Lugan...
5,Lausanne,NaN,NaN,CHE,Switzerland,"Lausanne, Switzerland",113.0,52.8,85.2,111.4,181.2,206.4,11.0,33.3,73.3,71.6,70.3,NaN,https://www.numbeo.com/cost-of-living/in/Lausa...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
620,Indore,NaN,NaN,IND,India,"Indore, India",18.1,4.0,11.6,20.5,68.7,NaN,NaN,NaN,NaN,51.8,NaN,60.2,https://www.numbeo.com/cost-of-living/in/Indor...
621,Guwahati,NaN,NaN,IND,India,"Guwahati, India",17.7,3.3,11.0,21.3,62.9,NaN,NaN,NaN,NaN,NaN,NaN,75.8,https://www.numbeo.com/cost-of-living/in/Guwah...
622,Lucknow,NaN,NaN,IND,India,"Lucknow (Lakhnau), India",17.5,3.4,11.0,19.8,69.8,NaN,NaN,NaN,NaN,54.5,62.9,77.9,https://www.numbeo.com/cost-of-living/in/Luckn...
